In [3]:
import pandas as pd
import numpy as np

url = 'https://eds-217-essential-python.github.io/data/marine_microplastics.csv'
df = pd.read_csv(url)

df = df.dropna(subset=['Measurement'])
samples = df[df['Unit'] == 'pieces/m3'].copy()
samples['ocean'] = samples['Oceans'].str.replace(' Ocean', '')
positive = samples[samples['Measurement'] > 0].copy()

positive.shape

(7091, 23)

In [4]:
positive.head()

,OBJECTID,Oceans,Regions,SubRegions,Sampling Method,Measurement,Unit,Density Range,Density Class,Short Reference,...,Keywords,Accession Number,Accession Link,Latitude,Longitude,Date,GlobalID,x,y,ocean
0,10008,Atlantic Ocean,NaN,NaN,Grab sample,0.020000,pieces/m3,0.005-1,Medium,Barrows et al.2018,...,Adventure Scientist/Citizen Science,211009,https://www.ncei.noaa.gov/access/metadata/land...,-58.428300,-64.1640,2/3/2017 12:00:00 AM,1e5b8e71-037b-4887-a276-f1e4552acb1f,-64.1640,-58.428300,Atlantic
1,8680,Atlantic Ocean,NaN,NaN,Grab sample,0.008000,pieces/m3,0.005-1,Medium,Barrows et al.2018,...,Adventure Scientist/Citizen Science,211009,https://www.ncei.noaa.gov/access/metadata/land...,-51.308200,-60.5467,11/17/2013 12:00:00 AM,a40f7f7c-1025-4aac-ad16-ee4cba196870,-60.5467,-51.308200,Atlantic
2,13257,Pacific Ocean,NaN,NaN,Manta net,0.019886,pieces/m3,0.005-1,Medium,Faure et al.2015,...,Oceaneye Association; Citizen Science,276422,https://www.ncei.noaa.gov/access/metadata/land...,-51.826667,-72.5750,12/26/2015 12:00:00 AM,febf79b8-7e2c-46e6-bc15-e08492ec2029,-72.5750,-51.826667,Pacific
3,9676,Atlantic Ocean,NaN,NaN,Grab sample,0.018000,pieces/m3,0.005-1,Medium,Barrows et al.2018,...,Adventure Scientist/Citizen Science,211009,https://www.ncei.noaa.gov/access/metadata/land...,-31.696000,-48.5600,8/11/2015 12:00:00 AM,a77121b2-e113-444e-82d9-7af11d62fdd2,-48.5600,-31.696000,Atlantic
5,10672,Pacific Ocean,NaN,NaN,Manta net,0.013000,pieces/m3,0.005-1,Medium,Goldstein et al.2013,...,Great Pacific Garbage Patch/SEAPLEX,253448,https://www.ncei.noaa.gov/access/metadata/land...,0.500000,-95.3500,10/17/2006 12:00:00 AM,23effcdd-35b7-4e1e-adb4-390693a287d3,-95.3500,0.500000,Pacific


In [5]:
positive.groupby('ocean')['Measurement'].mean().sort_values(ascending=False)

ocean
Pacific     595.199821
Atlantic    282.729748
Arctic        2.279608
Southern      0.075703
Name: Measurement, dtype: float64

In [6]:
positive.groupby('ocean')['Measurement'].median() 

ocean
Arctic      0.044000
Atlantic    0.019440
Pacific     0.116665
Southern    0.011500
Name: Measurement, dtype: float64

In [7]:
positive.groupby('ocean')['Measurement'].count() 

ocean
Arctic        58
Atlantic    6216
Pacific      799
Southern      18
Name: Measurement, dtype: int64

Refuse to report stats on Arctic and Southern Oceans because there is not enough data.

In [8]:
positive.groupby('ocean')['Measurement'].agg([
    'mean', 'median', 'count', 'max', 'min'])

,mean,median,count,max,min
ocean,,,,,
Arctic,2.279608,0.044000,58,63.000000,0.003000
Atlantic,282.729748,0.019440,6216,110480.000000,0.000676
Pacific,595.199821,0.116665,799,21156.558533,0.001000
Southern,0.075703,0.011500,18,0.499663,0.001000


A single large number typically scews the mean more than the median, therefore I would usually send the median. However, in this case, I think sending both values is important to demonstrate the range of values in the dataset.

In [9]:
positive.groupby('ocean').agg({
    'Measurement': 'mean', 
    'Sampling Method': 'nunique',
    'Organization': 'nunique'
    })

,Measurement,Sampling Method,Organization
ocean,,,
Arctic,2.279608,4,3
Atlantic,282.729748,10,20
Pacific,595.199821,7,9
Southern,0.075703,2,3


If one ocean had been sampled by only one organization using a single method, we could consider the data more standardized. Displaying the array of sampling methods and organizations demonstrates the amount of variability in sample collection methods. 

In [10]:
samp_method = positive.groupby('Sampling Method')['Measurement'].agg([
    'mean', 'median', 'count', 'max', 'min'])
samp_desc = samp_method.sort_values(
    'median', ascending=False).reset_index()
largest = samp_desc['median'][0]
print("Largest median= ", largest)

Largest median=  21840.0


In [11]:
samp_asc = samp_method.sort_values(
    'median', ascending=True).reset_index()
smallest = samp_asc['median'][0]
print("Smallest median= ", smallest)

Smallest median=  0.006


In [12]:
mag_diff = np.log10(largest/smallest)
mag_diff 

6.561101383649056

In a markdown cell, before you write any more code: if the oceans were not sampled with the same mix of methods, what might that imply regarding your answer from question 1? Two or three sentences.

In [13]:
ocean_samp = positive.groupby(['ocean', 'Sampling Method'])['Measurement'].count()
print(ocean_samp.sort_values(ascending=False).reset_index().head(10))
# 4390 NN samples in Atlantic, versus 158 in Pacific
print("Fraction P/A= ", 158/4390)

      ocean       Sampling Method  Measurement
0  Atlantic           Neuston net         4390
1  Atlantic           Grab sample          748
2  Atlantic             Manta net          631
3   Pacific             Manta net          226
4   Pacific          PVC cylinder          188
5   Pacific           Grab sample          166
6   Pacific           Neuston net          158
7  Atlantic  Intake seawater pump          106
8  Atlantic          PVC cylinder          104
9  Atlantic      Van Dorn sampler           84
Fraction P/A=  0.03599088838268793


In [23]:
nueston = positive[positive['Sampling Method'] == 'Neuston net'].copy()
nueston_grouped = nueston.groupby('ocean')['Measurement'].agg(
    ['mean', 'median', 'count'])
nueston_grouped

,mean,median,count
ocean,,,
Atlantic,0.085767,0.015120,4390
Pacific,0.396195,0.030680,158
Southern,0.278664,0.306271,4


In [24]:
positive_grouped = positive.groupby('ocean')['Measurement'].agg([
    'mean', 'median', 'count'])
positive_grouped

,mean,median,count
ocean,,,
Arctic,2.279608,0.044000,58
Atlantic,282.729748,0.019440,6216
Pacific,595.199821,0.116665,799
Southern,0.075703,0.011500,18


In [47]:
diff = ((positive_grouped['median'])-(nueston_grouped['median']))
print("Median difference between datasets")
print(diff)
compare_pos = (positive_grouped['median']['Atlantic'])-(positive_grouped['median']['Pacific'])
print("Median Difference in Q5 =", compare_pos)
compare_nues = (nueston_grouped['median']['Atlantic'])-(nueston_grouped['median']['Pacific'])
print("Median Difference btwn Nueston =", compare_nues)
counts_pos = (positive_grouped['count']['Atlantic'])-(positive_grouped['count']['Pacific'])
print("Count Difference in Q5 =", counts_pos)
counts_nues = (nueston_grouped['count']['Atlantic'])-(nueston_grouped['count']['Pacific'])
print("Count Difference btwn Nueston =", counts_nues)

Median difference between datasets
ocean
Arctic           NaN
Atlantic    0.004320
Pacific     0.085985
Southern   -0.294771
Name: median, dtype: float64
Median Difference in Q5 = -0.097225
Median Difference btwn Nueston = -0.01556
Count Difference in Q5 = 5417
Count Difference btwn Nueston = 4232


The Pacific median is larger than the Atlantic median in both the original and Nueston net dataset, however in both datasets, there are significantly more measurements taken in the Atlantic. 